# Advanced Forecasting Models

> **AUTHORITATIVE GENERATION — COMPLETE ADVANCED MODEL SET**  
> This executed notebook generates validated true rolling one-step vectors for ARIMA, Simple Exponential Smoothing (RMSE 1,855.73), and additive-trend non-seasonal Holt-Winters (RMSE 1,871.70), plus the Prophet 30-day periodic-refit vector. The smoothing results replace protocol-limited static RMSE values of 52,421.33 and 48,571.10. SARIMA is omitted because Step 4 found no supported weekly seasonality; PatchTST and iTransformer remain deferred to an isolated Python 3.12 environment.


## 1. Protocol and Environment

ARIMA uses a true rolling one-step protocol initialized from the final 128 training returns, then appends each observed return only after forecasting it. Simple Exponential Smoothing and additive-trend, non-seasonal Holt-Winters are refitted independently for every forecast date on the latest 128 strictly prior prices; each is therefore a true rolling one-step evaluation. Prophet uses a clearly labelled 30-day periodic-refit compromise with the latest 128 strictly prior observed prices at every refit; it is not described as rolling one-step.

The Step 4 diagnostic found lag-7 ACF `-0.0234`, within the approximate 95% bound `±0.0269`, and weekly STL seasonal strength `0.070`. Therefore no weekly seasonal component is used in Holt-Winters or SARIMA. A separate SARIMA result is omitted because `seasonal_order=(0,0,0,0)` would reduce to a non-seasonal ARIMA/SARIMAX specification and would not provide distinct seasonal evidence.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.bitcoin_advanced_models import (
    generate_exponential_smoothing_validation_artifacts,
    merge_advanced_forecasts,
    periodic_refit_prophet_forecast,
    rolling_arima_log_return_forecast,
    rolling_holt_winters_forecast,
    rolling_simple_exp_smoothing_forecast,
    validate_and_save_forecast,
)
from src.data_loader import load_bitcoin_data
from src.preprocessing import prepare_daily_bitcoin_data
from src.metrics import mae, rmse, mape, smape
pd.set_option('display.float_format', '{:.6f}'.format)

E:\Re_Sc_AM\TimeSeriesFoundationModels\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Importing plotly failed. Interactive plots will not work.


## 2. Load Dataset and Freeze Split

In [2]:
data_path = PROJECT_ROOT / 'data' / 'bitcoin' / 'btcusd_1-min_data.csv'
raw = load_bitcoin_data(data_path)
daily = prepare_daily_bitcoin_data(raw)
target = daily['Close'].asfreq('D').dropna()
split_index = int(len(target) * 0.8)
train = target.iloc[:split_index]
test = target.iloc[split_index:]
assert len(train) == 4241 and len(test) == 1061
assert train.index.max() < test.index.min()
pd.DataFrame({'Segment':['Train','Test'], 'Rows':[len(train),len(test)], 'Start':[train.index.min(),test.index.min()], 'End':[train.index.max(),test.index.max()]})

,Segment,Rows,Start,End
0,Train,4241,2012-01-01 00:00:00+00:00,2023-08-11 00:00:00+00:00
1,Test,1061,2023-08-12 00:00:00+00:00,2026-07-07 00:00:00+00:00


## 3. Rolling One-Step ARIMA on Log Returns

In [3]:
arima_forecast = rolling_arima_log_return_forecast(target, test.index, context_length=128, order=(1,0,1))
arima_frame = validate_and_save_forecast(arima_forecast, test.index, PROJECT_ROOT / 'results' / 'arima_rolling_forecast.csv')
arima_metrics = pd.DataFrame([{'Model':'ARIMA Rolling One-Step','MAE':mae(test,arima_forecast),'RMSE':rmse(test,arima_forecast),'MAPE':mape(test,arima_forecast),'sMAPE':smape(test,arima_forecast)}]).set_index('Model')
arima_metrics

,MAE,RMSE,MAPE,sMAPE
Model,,,,
ARIMA Rolling One-Step,1299.874638,1866.302859,1.754004,1.754209


## 4. SARIMA Decision

No separate SARIMA model is generated. The empirical weekly-seasonality diagnostics do not support a seven-day term, and a SARIMA specification with `seasonal_order=(0,0,0,0)` would provide no seasonal mechanism and no distinct forecast vector beyond a non-seasonal ARIMA/SARIMAX model.

## 5. Prophet 30-Day Periodic Refit

In [4]:
prophet_forecast = periodic_refit_prophet_forecast(target, test.index, context_length=128, refit_every=30)
prophet_frame = validate_and_save_forecast(prophet_forecast, test.index, PROJECT_ROOT / 'results' / 'prophet_rolling_forecast.csv')
prophet_metrics = pd.DataFrame([{'Model':'Prophet 30-Day Periodic Refit','MAE':mae(test,prophet_forecast),'RMSE':rmse(test,prophet_forecast),'MAPE':mape(test,prophet_forecast),'sMAPE':smape(test,prophet_forecast)}]).set_index('Model')
prophet_metrics

08:29:45 - cmdstanpy - INFO - Chain [1] start processing


08:29:45 - cmdstanpy - INFO - Chain [1] done processing


08:29:46 - cmdstanpy - INFO - Chain [1] start processing


08:29:46 - cmdstanpy - INFO - Chain [1] done processing


08:29:46 - cmdstanpy - INFO - Chain [1] start processing


...

08:30:02 - cmdstanpy - INFO - Chain [1] start processing


08:30:02 - cmdstanpy - INFO - Chain [1] done processing


08:30:03 - cmdstanpy - INFO - Chain [1] start processing


08:30:03 - cmdstanpy - INFO - Chain [1] done processing


,MAE,RMSE,MAPE,sMAPE
Model,,,,
Prophet 30-Day Periodic Refit,8195.262862,10781.162873,11.199767,11.287185


## 6. Rolling One-Step Simple Exponential Smoothing

The model is refitted for every test date on the latest 128 prices ending strictly before that date.


In [5]:
simple_exp_smoothing_forecast = rolling_simple_exp_smoothing_forecast(target, test.index, context_length=128)
simple_exp_smoothing_frame = validate_and_save_forecast(simple_exp_smoothing_forecast, test.index, PROJECT_ROOT / 'results' / 'simple_exp_smoothing_forecast.csv')
simple_exp_smoothing_metrics = pd.DataFrame([{'Model':'Simple Exponential Smoothing Rolling One-Step','MAE':mae(test,simple_exp_smoothing_forecast),'RMSE':rmse(test,simple_exp_smoothing_forecast),'MAPE':mape(test,simple_exp_smoothing_forecast),'sMAPE':smape(test,simple_exp_smoothing_forecast)}]).set_index('Model')
simple_exp_smoothing_metrics


,MAE,RMSE,MAPE,sMAPE
Model,,,,
Simple Exponential Smoothing Rolling One-Step,1290.358684,1855.731424,1.742685,1.743871


## 7. Rolling One-Step Holt-Winters Exponential Smoothing

An additive trend and no seasonal component are used. The model is refitted for every test date on the latest 128 strictly prior prices.


In [6]:
holt_winters_forecast = rolling_holt_winters_forecast(target, test.index, context_length=128)
holt_winters_frame = validate_and_save_forecast(holt_winters_forecast, test.index, PROJECT_ROOT / 'results' / 'holt_winters_forecast.csv')
holt_winters_metrics = pd.DataFrame([{'Model':'Holt-Winters Rolling One-Step','MAE':mae(test,holt_winters_forecast),'RMSE':rmse(test,holt_winters_forecast),'MAPE':mape(test,holt_winters_forecast),'sMAPE':smape(test,holt_winters_forecast)}]).set_index('Model')
holt_winters_metrics


,MAE,RMSE,MAPE,sMAPE
Model,,,,
Holt-Winters Rolling One-Step,1308.541314,1871.702185,1.763640,1.763424


## 8. Pre-Test Validation Forecasts for Empirical Uncertainty

The same strictly prior rolling protocol is run over the final 1,061 training dates. These residuals provide leakage-free empirical uncertainty scores in Notebook 06.


In [7]:
simple_exp_smoothing_validation_frame, holt_winters_validation_frame = generate_exponential_smoothing_validation_artifacts(PROJECT_ROOT)
assert simple_exp_smoothing_validation_frame['Timestamp'].max() < test.index.min()
assert holt_winters_validation_frame['Timestamp'].max() < test.index.min()


E:\Re_Sc_AM\TimeSeriesFoundationModels\.venv\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


E:\Re_Sc_AM\TimeSeriesFoundationModels\.venv\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


## 9. Deferred NeuralForecast Models

PatchTST and iTransformer via neuralforecast were not run in this environment. neuralforecast>=3.1.8 requires pytorch-lightning<2.6.0 (incompatible with the project's pinned lightning 2.6.5); neuralforecast==3.1.7 is compatible with lightning but imports ray[train,tune], which has no available Windows/Python 3.13 distribution in this environment as of this run. A separate Python 3.12 environment (the same fix already planned for Moirai/uni2ts) is the recommended path if these models are needed later.

## 10. Merge and Validate Artifacts

In [8]:
validated_path = PROJECT_ROOT / 'results' / 'validated_forecasts.csv'
advanced_frames = [arima_frame, prophet_frame, simple_exp_smoothing_frame, holt_winters_frame]
validated = merge_advanced_forecasts(validated_path, advanced_frames)
model_columns = [frame.columns[1] for frame in advanced_frames]
checks = pd.DataFrame({
    'Check':['All vector rows','All vectors aligned','No NaNs','Finite values','No duplicate vectors'],
    'Pass':[all(len(frame)==1061 for frame in advanced_frames),all(pd.DatetimeIndex(frame.Timestamp).equals(test.index) for frame in advanced_frames),not validated.isna().any().any(),np.isfinite(validated.select_dtypes(include=[np.number])).all().all(),not validated[model_columns].T.duplicated().any()]
})
assert checks['Pass'].all()
checks


,Check,Pass
0,All vector rows,True
1,All vectors aligned,True
2,No NaNs,True
3,Finite values,True
4,No duplicate vectors,True


## 11. Final Numeric Comparison

In [9]:
comparison = pd.concat([arima_metrics, prophet_metrics, simple_exp_smoothing_metrics, holt_winters_metrics]).sort_values('RMSE')
comparison


,MAE,RMSE,MAPE,sMAPE
Model,,,,
Simple Exponential Smoothing Rolling One-Step,1290.358684,1855.731424,1.742685,1.743871
ARIMA Rolling One-Step,1299.874638,1866.302859,1.754004,1.754209
Holt-Winters Rolling One-Step,1308.541314,1871.702185,1.763640,1.763424
Prophet 30-Day Periodic Refit,8195.262862,10781.162873,11.199767,11.287185
